# Drifter Data Cleaning Pipeline

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import netCDF4 as nc
from pathlib import Path
from scipy.signal import butter, sosfiltfilt, windows

In [2]:
# Paths
input_data = Path("/home/rosquete/Documents/FRESH-CARE/data/fusion_evaluation/classic_evaluation/in_situ_data/")
data_per_bouy = input_data / Path("data_per_bouy/")
data_filtered = input_data / Path("data_filtered/")
data_ready = input_data / Path("data_ready/")

In [3]:
# Processing constants
CHUNK = 20000
FILL = {-1e34, -999999}
cols = ["time", "latitude", "lon360", "ve", "vn", "err_lat", "err_lon"]
ERR_POS_MAX = 0.007
MAX_SPEED = 2.5               # m/s
FS = 4.0                      # d^-1
NYQ = FS / 2                  # Nyquist frequency: 2.0 cycles/day
CUTOFF_PERIOD = 1.5           # 36 hours expressed in days (36/24)
Wn = (1/CUTOFF_PERIOD) / NYQ  # normalized cutoff frequency (~0.333)
ORDER = 4                     # 4th order Butterworth for sharp decay
CUT = 4                       # edge points to remove after filtering (4 * 6h = 24h)

In [4]:
# Load input NetCDF file
nc_file = input_data / Path("raw_data/drifter_6hour_360lon.nc")

In [5]:
# Extract and separate data by buoy

def dec(txt):
    """Decode string variables from NetCDF"""
    arr = ds[txt][start:end]
    return np.char.strip(nc.chartostring(arr) if arr.ndim > 1 else arr.astype(str))

def manage_times(ds, chunk):
    """Extract deployment and drogue lost dates"""
    deploy = {}
    drogue = {}
    n = ds.dimensions["row"].size

    for start in range(0, n, chunk):
        end = min(start + chunk, n)

        # Extract IDs and timestamps
        arr_id = ds["ID"][start:end]
        ids = np.char.strip(nc.chartostring(arr_id) if arr_id.ndim > 1 else arr_id.astype(str))

        dep = ds["deploy_date"][start:end].filled(np.nan)
        drg = ds["drogue_lost_date"][start:end].filled(np.nan)

        for i, bid in enumerate(ids):
            if bid not in deploy and not np.isnan(dep[i]):
                deploy[bid] = dep[i]
            if bid not in drogue and not np.isnan(drg[i]):
                drogue[bid] = drg[i]

    return deploy, drogue

# Process NetCDF file in chunks
with nc.Dataset(nc_file) as ds:
    rows = ds.dimensions["row"].size  
    deploy_dict, drogue_dict = manage_times(ds, CHUNK)

    for start in range(0, rows, CHUNK):
        end = min(start + CHUNK, rows)

        # Extract metadata
        ids = dec("ID")
        wmo = dec("WMO")
        typeb = dec("typebuoy")

        # Extract data variables
        datos = {c: ds[c][start:end].filled(np.nan) for c in cols}
        df = pd.DataFrame(datos)
        df["ID"] = ids
        df["WMO"] = wmo
        df["typeb"] = typeb
        df["time"] = pd.to_datetime(df["time"], unit="s", origin="1970-01-01")

        # Save data by buoy ID 
        for buoy_id, g in df.groupby("ID"):
            name = buoy_id.strip()
            path = data_per_bouy / f"{name}.csv"
            is_new = not path.exists()
            dep = deploy_dict.get(buoy_id)
            drg = drogue_dict.get(buoy_id)
            with open(path, "a") as f:
                if is_new:
                    f.write(f"ID,{name}\n")    
                    f.write(f"WMO,{g['WMO'].iloc[0]}\n")       
                    f.write(f"typeboy,{g['typeb'].iloc[1]}\n")  
                    # f.write(f"typeloc,{g['typeloc'].iloc[2]}\n")
                    if dep is not None:
                        f.write(f"deploy_date,{pd.to_datetime(dep, unit='s', utc=True)}\n")
                    if drg is not None:
                        f.write(f"drogue_lost_date,{pd.to_datetime(drg, unit='s', utc=True)}\n")
                g[cols].to_csv(f, index=False, header=is_new)
        print(f"{end}/{rows} processed rows")

20000/1909488 processed rows
40000/1909488 processed rows
60000/1909488 processed rows
80000/1909488 processed rows
60000/1909488 processed rows
80000/1909488 processed rows
100000/1909488 processed rows
120000/1909488 processed rows
100000/1909488 processed rows
120000/1909488 processed rows
140000/1909488 processed rows
160000/1909488 processed rows
140000/1909488 processed rows
160000/1909488 processed rows
180000/1909488 processed rows
200000/1909488 processed rows
180000/1909488 processed rows
200000/1909488 processed rows
220000/1909488 processed rows
240000/1909488 processed rows
260000/1909488 processed rows
220000/1909488 processed rows
240000/1909488 processed rows
260000/1909488 processed rows
280000/1909488 processed rows
300000/1909488 processed rows
320000/1909488 processed rows
280000/1909488 processed rows
300000/1909488 processed rows
320000/1909488 processed rows
340000/1909488 processed rows
360000/1909488 processed rows
380000/1909488 processed rows
340000/1909488 p

In [6]:
# Apply quality filters to buoy data

for csv_path in data_per_bouy.glob("*.csv"):
    # Read metadata and find header
    meta_lines, header_idx = [], None
    with open(csv_path) as f:
        for i, line in enumerate(f):
            if line.lstrip().lower().startswith("time,"):
                header_idx = i
                break
            meta_lines.append(line.strip())

    if header_idx is None:
        print(f"Skipping {csv_path.name}: no time header")
        continue

    meta = dict(l.split(",", 1) for l in meta_lines if "," in l)

    # Filter for SVP buoys only
    tb = meta.get("typebuoy") or meta.get("typeboy") or ""
    tb = tb.strip().upper()

    if "SVP" not in tb:
        print(f"Skipping {csv_path.name}: not SVP buoy ({tb})")
        continue

    df = pd.read_csv(csv_path, skiprows=header_idx, low_memory=False)

    # Convert velocity components to numeric
    for c in ["ve", "vn", "err_lat", "err_lon"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # Remove missing velocity data
    df = df.dropna(subset=["ve", "vn"])
    df = df[~df["ve"].isin(FILL) & ~df["vn"].isin(FILL)]

    # Convert timestamps and filter for drogue-on period
    df["time"] = pd.to_datetime(df["time"], utc=True, errors="coerce")
    df = df[df["time"].notna()]

    d_meta = meta.get("drogue_lost_date")
    if d_meta:
        d_dt = pd.to_datetime(d_meta, utc=True, errors="coerce")
        if not pd.isna(d_dt):
            df = df[df["time"] <= d_dt]

    # Filtering by typeloc proxy (error in position, ARGOS vs GPS)
    if "err_lat" in df.columns and "err_lon" in df.columns:
        df = df[(df["err_lat"] <= ERR_POS_MAX) & (df["err_lon"] <= ERR_POS_MAX)]

    # speed <= MAX_SPEED
    df["speed"] = np.sqrt(df["ve"]**2 + df["vn"]**2)
    df = df[df["speed"] <= MAX_SPEED]

    if df.empty:
        print(f"Skipping {csv_path.name}: empty after filtering")
        continue

    out = data_filtered / csv_path.name
    with open(out, "w") as f:
        f.write("\n".join(meta_lines) + ("\n" if meta_lines else ""))
        df.drop(columns=["speed"]).to_csv(f, index=False)

    print(f"OK {csv_path.name} -> {out.name} ({len(df)} rows)")

Skipping 89804.csv: empty after filtering
OK 104126.csv -> 104126.csv (249 rows)
OK 300234064874070.csv -> 300234064874070.csv (1380 rows)
OK 300234068446260.csv -> 300234068446260.csv (938 rows)
Skipping 300234068441310.csv: empty after filtering
Skipping 300234064503560.csv: empty after filtering
OK 300234066512810.csv -> 300234066512810.csv (407 rows)
OK 300234067874490.csv -> 300234067874490.csv (85 rows)
Skipping 89888.csv: empty after filtering
OK 300234066414750.csv -> 300234066414750.csv (508 rows)
OK 300234062887800.csv -> 300234062887800.csv (1091 rows)
OK 300534061759270.csv -> 300534061759270.csv (272 rows)
OK 300234010162380.csv -> 300234010162380.csv (269 rows)
OK 300534061181960.csv -> 300534061181960.csv (119 rows)
OK 300234066218670.csv -> 300234066218670.csv (224 rows)
OK 132488.csv -> 132488.csv (1485 rows)
OK 300234060891140.csv -> 300234060891140.csv (1719 rows)
OK 83468.csv -> 83468.csv (164 rows)
Skipping 300534061600990.csv: empty after filtering
OK 300234061668

In [7]:
# Final data processing: filtering and daily averaging

# Create low-pass filter
sos = butter(ORDER, Wn, btype='low', output='sos')

def process_6hourly(df, name):
    """Standardize data to 6-hour grid"""
    df = df[df["time"].notna()]
    
    # Round to 6-hour intervals
    df["time"] = df["time"].dt.round("6h")  
    df = df.sort_values("time").drop_duplicates("time")

    # Check minimum data coverage
    total_duration = (df['time'].max() - df['time'].min()).total_seconds()
    if total_duration < 4 * 24 * 3600:
        print(f"Warning: Less than 4 days of data in {name}")
        return None

    # Check for large gaps
    gaps = df["time"].diff().dt.total_seconds().dropna()
    if not gaps.empty and gaps.max() > 18 * 3600:
        print(f"Warning {name}: Large gaps detected, skipping")
        return None

    # Create regular 6-hour grid
    idx = pd.date_range(df["time"].iloc[0], df["time"].iloc[-1], freq="6h")
    df_resampled = df.set_index("time").reindex(idx)

    # Interpolate missing values
    interp_cols = ["latitude","lon360","ve","vn","err_lat","err_lon"]
    df_resampled[interp_cols] = df_resampled[interp_cols].interpolate(method="time", limit=2)
    
    return df_resampled

def low_pass_filter(series):
    """Apply low-pass Butterworth filter"""
    # Zero-phase filtering to avoid time shifts
    xf = sosfiltfilt(sos, series)
    return xf

# Process all filtered files
for csv in data_filtered.glob("*.csv"):
    # Read metadata and data
    meta_lines = []
    header_idx = None
    with open(csv) as f:
        for i, line in enumerate(f):
            if line.lstrip().lower().startswith("time,"):
                header_idx = i
                break
            meta_lines.append(line.rstrip("\n"))
    if header_idx is None:
        print(f"Warning {csv.name}: no time header")
        continue
    
    df = pd.read_csv(csv, skiprows=header_idx, low_memory=False, parse_dates=["time"])
    
    # Standardize to 6-hour grid
    df_tmp = process_6hourly(df, csv.name)
    if df_tmp is None or len(df_tmp) <= 2 * CUT:
        continue

    # Apply low-pass filter to velocities
    ve_f = low_pass_filter(df_tmp["ve"].values)
    vn_f = low_pass_filter(df_tmp["vn"].values)

    # Remove filter edge effects
    df_tmp = df_tmp.iloc[CUT:-CUT].copy()
    df_tmp["ve"] = ve_f[CUT:-CUT]
    df_tmp["vn"] = vn_f[CUT:-CUT]

    # Create daily averages
    # lon360 is circular: average via sin/cos to avoid 0°/360° wrap artefacts
    df_tmp["_lon_sin"] = np.sin(np.radians(df_tmp["lon360"]))
    df_tmp["_lon_cos"] = np.cos(np.radians(df_tmp["lon360"]))

    g = df_tmp.resample("1D")
    counts = g.size()
    df_daily = g.mean(numeric_only=True)

    # Require at least 3 of 4 points for daily value
    df_daily = df_daily[counts >= 3].reset_index().rename(columns={"index": "time"})

    # Reconstruct lon360 from circular mean and drop helper columns
    df_daily["lon360"] = np.degrees(np.arctan2(df_daily["_lon_sin"], df_daily["_lon_cos"])) % 360
    df_daily = df_daily.drop(columns=["_lon_sin", "_lon_cos"])

    # Calculate speed and direction
    df_daily["speed"] = np.sqrt(df_daily["ve"]**2 + df_daily["vn"]**2)
    df_daily["direction"] = (np.degrees(np.arctan2(df_daily["ve"], df_daily["vn"])) + 360) % 360


    # Save processed data
    out = data_ready / csv.name
    with open(out, "w") as f:
        f.write("\n".join(meta_lines) + "\n")
        df_daily.to_csv(f, index=False)

    print(f"OK -> {csv.name} (Daily processed)")


OK -> 104126.csv (Daily processed)
Warning 300234064874070.csv: Large gaps detected, skipping
Warning 300234068446260.csv: Large gaps detected, skipping
OK -> 300234066512810.csv (Daily processed)
Warning 300234067874490.csv: Large gaps detected, skipping
OK -> 300234066414750.csv (Daily processed)
Warning 300234062887800.csv: Large gaps detected, skipping
OK -> 300534061759270.csv (Daily processed)
Warning 300234010162380.csv: Large gaps detected, skipping
OK -> 300534061181960.csv (Daily processed)
Warning 300234066218670.csv: Large gaps detected, skipping
OK -> 132488.csv (Daily processed)
Warning 300234060891140.csv: Large gaps detected, skipping
OK -> 83468.csv (Daily processed)
Warning 300234061668320.csv: Large gaps detected, skipping
OK -> 300234062857970.csv (Daily processed)
OK -> 300234066516010.csv (Daily processed)
OK -> 300234065800030.csv (Daily processed)
Warning 300234064506400.csv: Large gaps detected, skipping
OK -> 300234066394370.csv (Daily processed)
OK -> 3002340